In [0]:
%run "../utils/config_loader"

In [0]:
# Cell 1 — Widget
dbutils.widgets.dropdown("environment", "dev", ["dev", "preprod", "prod"], "Environment")



# Cell 3 — Load config
config = load_config_from_widget()
print(f"🚀 Bronze Pharmacy Pipeline - {config['environment'].upper()} Environment")
print(f"Status: Running bronze ingestion...")
print(f"📂 Reading from : {config['storage']['landing']['pharmacy']}")
print(f"📂 Writing to   : {config['storage']['external']['bronze']}")

if config['environment'] == 'prod':
    raise Exception("⛔ You are on PROD! Switch to dev first!")
elif config['environment'] == 'preprod':
    print("⚠️  WARNING: Running on PREPROD!")
else:
    print("✅ Running on DEV — safe to proceed!")

# Cell 4 — Imports
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Cell 5 — Read raw CSV
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv(config['storage']['landing']['pharmacy'])

display(df_raw)
print(f"📊 Raw rows: {df_raw.count()}")

# Cell 6 — Schema validation
expected_columns = ["sale_id", "store_id", "store_name", "drug_id",
                    "drug_name", "category", "quantity", "unit_price",
                    "sale_date", "region", "pharmacist_id"]

missing_cols = set(expected_columns) - set(df_raw.columns)

print("=== SCHEMA VALIDATION ===")
if missing_cols:
    raise Exception(f"Schema FAILED! Missing: {missing_cols}")
else:
    print("✅ Schema validation PASSED!")

# Cell 7 — Add metadata
df_bronze = df_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("environment",         lit(config['environment'])) \
    .withColumn("source_file",         lit(config['storage']['landing']['pharmacy']))

# Cell 8 — Save to bronze
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{config['schemas']['bronze']}")

external_path = f"{config['storage']['external']['bronze']}/pharmacy_raw"
bronze_table  = f"{config['catalog']}.{config['schemas']['bronze']}.pharmacy_raw"

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .option("path", external_path) \
    .saveAsTable(bronze_table)

print(f"\n=== BRONZE COMPLETE ===")
print(f"🌍 Environment : {config['environment'].upper()}")
print(f"📋 Table       : {bronze_table}")
print(f"📂 Path        : {external_path}")
print(f"📊 Rows        : {spark.table(bronze_table).count()}")

In [0]:
dbutils.notebook.run(
    "/Workspace/Users/lakshmisas96@gmail.com/walgreens-pharmacy-pipeline/walgreens-pharmacy-pipeline/utils/config_loader",
    60
)